# [14.1] JEPA and World-Model Controls - Exercises

**Core question.** When does a frozen video latent deserve a world-state interpretation?

You will build the toy controls before reading the V-JEPA 2 report:
paired target prediction, non-collapse, held-out state probes, transition
baselines, and object permanence controls. The final section reads a pinned
CUDA report generated from `facebook/vjepa2-vitl-fpc64-256` on deterministic
generated videos.

## Learning Objectives

- Implement paired cosine target checks.
- Reject collapsed or wrong-scale embedding evidence.
- Compare held-out state probes against shuffled-label controls.
- Compare latent rollouts against copy and shuffled-action baselines.
- Compare occluded-object evidence against absent and different-object controls.

> ```yaml
> Difficulty: 4
> Importance: 4
> ```

<details>
<summary>Help - what makes this ARENA-style?</summary>

Every claim has a toy oracle and a matched control. If a metric only looks
pretty but does not beat the control, it is a negative result.

</details>

In [ ]:
GT_TIER = "GT-1"
EXERCISE_ID = "14_1_jepa_and_world_model_controls"
DIFFICULTY = 4
IMPORTANCE = 4
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA V-JEPA 2 latent-control preflight"
REQUIRES_GPU = True

import json
import sys
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter14_jepa_world_models"
section = "part1_jepa_world_model_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_jepa_world_model_controls.tests as tests
import part1_jepa_world_model_controls.utils as utils

from arena_ext.jepa_world_models import (
    collapse_diagnostics_report,
    jepa_prediction_report,
    latent_rollout_report,
    object_permanence_report,
    transition_consistency_report,
    world_state_probe_report,
)

MAIN = True

## Exercise 1 - Paired Cosine

Implement row-wise cosine similarity for matching predicted and target embeddings.

<details>
<summary>Expected output</summary>

```text
All tests in `test_paired_cosine_toy_oracle` passed!
All tests in `test_paired_cosine_rejects_shape_mismatch` passed!
```

</details>

<details>
<summary>Help - why shape errors matter</summary>

Broadcasting target embeddings can make a broken predictor look plausible. Reject mismatched shapes before computing scores.

</details>

<details>
<summary>Solution</summary>

Use `float()` tensors, multiply row-wise, divide by the product of row norms,
and clamp the denominator by `eps`. Raise `ValueError` for shape mismatches.

</details>

In [ ]:
def paired_cosine(left: t.Tensor, right: t.Tensor, *, eps: float = 1e-8) -> t.Tensor:
    """
    Return row-wise cosine similarities.

    Inputs:
        left: shape [..., d_model]
        right: shape [..., d_model]
    Returns:
        cosine: shape [...]
    """
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_paired_cosine_toy_oracle(paired_cosine)
tests.test_paired_cosine_rejects_shape_mismatch(paired_cosine)

## Exercise 2 - JEPA Target Prediction and Collapse

Build a passing target-prediction report, then add a non-collapse check that rejects identical features.

<details>
<summary>Expected output</summary>

```text
mean_cosine: 1.0
mse: 0.0
collapsed_control_rejected: True
```

</details>

<details>
<summary>Help - why use cosine and MSE?</summary>

Cosine checks direction; MSE catches wrong scale. A target predictor needs both to pass.

</details>

<details>
<summary>Solution</summary>

Use exact one-hot target embeddings for the positive case. For collapse, compare
a small structured feature matrix against an all-ones matrix and require the
all-ones matrix to fail `collapse_diagnostics_report`.

</details>

In [ ]:
def jepa_prediction_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def collapse_diagnostics_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_jepa_prediction_smoke_test(jepa_prediction_smoke_test)
tests.test_jepa_prediction_report_rejects_collapse_and_bad_mse()
tests.test_collapse_diagnostics_smoke_test(collapse_diagnostics_smoke_test)
tests.test_collapse_diagnostics_rejects_identical_features()

## Exercise 3 - Held-Out State Probes

Use aligned toy logits to pass, then show that shuffled labels fail.

<details>
<summary>Expected output</summary>

```text
accuracy: 1.0
shuffled_control_rejected: True
accuracy_margin: 1.0
```

</details>

<details>
<summary>Help - what this probe does not prove</summary>

A probe shows decodable information. It does not prove the model uses that information causally.

</details>

<details>
<summary>Solution</summary>

Pass aligned labels to `world_state_probe_report`, then pass deliberately
swapped labels to the same logits. The aligned report should pass and the
shuffled-label report should fail.

</details>

In [ ]:
def state_probe_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def state_probe_control_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_state_probe_smoke_test(state_probe_smoke_test)
tests.test_state_probe_control_smoke_test(state_probe_control_smoke_test)
tests.test_state_probe_report_rejects_shuffled_labels()

## Exercise 4 - Transition and Rollout Controls

Test `state + action_delta = next_state`, then require a rollout loss to beat copy and shuffled-action baselines.

<details>
<summary>Expected output</summary>

```text
mean_cosine: 1.0
rollout_passes: True
copy_and_shuffled_controls_rejected: True
```

</details>

<details>
<summary>Help - why copy is a strong baseline</summary>

If adjacent states are similar, copying the current latent may look good. A world-model claim needs an action-conditioned advantage.

</details>

<details>
<summary>Solution</summary>

Use the exact toy identity `state + action_delta == next_state` for the tensor
check. For the rollout check, set the rollout loss below both the copy and
shuffled-action baselines, then include a failed control that does not beat them.

</details>

In [ ]:
def transition_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def rollout_control_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_transition_smoke_test(transition_smoke_test)
tests.test_transition_report_rejects_missing_action_delta()
tests.test_rollout_control_smoke_test(rollout_control_smoke_test)
tests.test_latent_rollout_report_rejects_copy_and_shuffled_controls()

## Exercise 5 - Object Permanence Controls

An occluded object should remain above absent-object evidence, and different-object similarity should not count as permanence.

<details>
<summary>Expected output</summary>

```text
occluded_absent_gap: 0.575
absent_like_rejected: True
different_object_rejected: True
```

</details>

<details>
<summary>Help - what the absent control catches</summary>

An occluder or background can make videos similar even when the object is absent. This control asks for object-specific evidence.

</details>

<details>
<summary>Solution</summary>

Make the positive occluded score high and the absent score low. Then include two
negative controls: an occluded score close to absent, and a high score that is
also high for a different object.

</details>

In [ ]:
def object_permanence_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def object_permanence_control_smoke_test() -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_object_permanence_smoke_test(object_permanence_smoke_test)
tests.test_object_permanence_control_smoke_test(object_permanence_control_smoke_test)
tests.test_object_permanence_report_rejects_absent_and_different_object_controls()

## Exercise 6 - Notebook Contract

Collect the toy checks into one JSON-serializable contract. This is the CPU teaching path; repository acceptance still reruns the real CUDA report.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Solution</summary>

Return a dictionary with every toy report: target prediction, collapse,
state-probe control, transition, rollout control, object permanence, and
object-permanence controls.

</details>

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_notebook_contract(run_smoke_test)

## Signature Result

The committed CUDA report is the section's signature result. Pinned V-JEPA 2
frozen latents pass generated-video controls for target prediction, non-collapse,
state probes, action-conditioned rollout, object permanence, and object-token
patching.

| Check | Observed | Required |
|---|---:|---:|
| Same-object margin | `0.0488` | `>= 0.030` |
| Occluded-vs-absent gap | `0.2230` | `>= 0.200` |
| Masked prediction loss reduction | `0.9992` | `>= 0.500` |
| Probe margin over shuffled | `0.5400` | `>= 0.200` |
| Object-token patch gap | `0.99997` | `>= 0.400` |

<details>
<summary>Help - why read a committed report?</summary>

The notebook is usable on CPU-only machines, but the repository acceptance gate reruns the full V-JEPA 2 CUDA path. Here you inspect the report and verify that it contains the controls it claims.

</details>

## Limitations

### What this does not show

This is not a real-video object-permanence benchmark, not V-JEPA fine-tuning, and not a replication of I-JEPA, VL-JEPA, Othello-GPT, maze, Sudoku, or RL world models.

## Bonus - Anomaly Hunting

- Move the occluder and check whether the absent-object control becomes too easy.
- Shuffle actions within the same displacement magnitude and rerun the rollout control.
- Patch background tokens instead of object tokens and compare the position-probe effect.

In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"]
    assert gpu["cuda_available"]
    assert gpu["vjepa2_preflight_passed"]
    assert gpu["vjepa2_world_model_controls_passed"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_committed_verification_report_vjepa_world_controls()
tests.test_exercise_notebook_declares_full_verification_contract()